# Лабораторная работа — Car Price Prediction

**Задача:** разведочный анализ данных (EDA) перед прогнозированием стоимости автомобиля (регрессия).

**Исходные столбцы (строго 8):** `name`, `year`, `selling_price`, `km_driven`, `fuel`, `seller_type`, `transmission`, `owner`.

**Целевая переменная:** `selling_price` (цена продажи, INR). Новые признаки не создаются.

**План:** загрузка → знакомство → очистка → 5 графиков (4 PNG и 1 интерактивный HTML) → сохранение очищенных данных → выводы для README.

> Запускайте ячейки сверху вниз. Положите CSV в `data/dataset.csv`. Рабочая папка — корень проекта или `eda/`.

In [ ]:
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from IPython.display import display, Markdown

# Отображение таблиц и графиков
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 30)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 150

# Фиксированный список столбцов: переменную columns НЕ используем.
SOURCE_COLUMNS = (
    'name', 'year', 'selling_price', 'km_driven',
    'fuel', 'seller_type', 'transmission', 'owner'
)

# В VS Code рабочей папкой может оказаться корень проекта или папка eda.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / 'data' / 'dataset.csv').is_file()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Не найден data/dataset.csv. Поместите туда исходный CSV и откройте папку проекта в VS Code.'
    )
PROJECT_ROOT = PROJECT_ROOT.resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'dataset.csv'
EDA_DIR = PROJECT_ROOT / 'eda'
EDA_DIR.mkdir(parents=True, exist_ok=True)

print('Корневая директория:', PROJECT_ROOT)
print('Файл с данными:', DATA_PATH)

## 1. Загрузка и знакомство с данными

Проверим состав данных, объём выборки, типы переменных, статистики, пропуски и дубликаты. Используем **только 8 выбранных исходных столбцов**, даже если CSV содержит дополнительные.

In [ ]:
# Загрузка CSV и нормализация названий колонок
raw = pd.read_csv(DATA_PATH)
raw.columns = raw.columns.str.strip()

absent = [col for col in SOURCE_COLUMNS if col not in raw.columns]
if absent:
    raise ValueError(f'В CSV отсутствуют требуемые столбцы: {absent}')

df = raw.loc[:, list(SOURCE_COLUMNS)].copy()
print(f'Объектов: {len(df)}; столбцов: {df.shape[1]}')
display(df.head())

print('Типы данных:')
df.info()

print('Описательная статистика:')
display(df.describe(include='all').T)

print('Пропуски в исходных данных:')
display(df.isna().sum().to_frame('Пропуски'))
print('Полные дубликаты:', int(df.duplicated().sum()))

In [ ]:
# Делим исходные признаки на числовые, категориальные и целевой.
TARGET = 'selling_price'
NUMERIC_FEATURES = ['year', 'km_driven']
CATEGORICAL_FEATURES = ['name', 'fuel', 'seller_type', 'transmission', 'owner']

print('Целевая переменная:', TARGET)
print('Числовые входные признаки:', NUMERIC_FEATURES)
print('Категориальные входные признаки:', CATEGORICAL_FEATURES)

for feature in ['fuel', 'seller_type', 'transmission', 'owner']:
    print(f'\nКоличество записей по категориям: {feature}')
    display(df[feature].value_counts(dropna=False).to_frame('Количество'))

## 2. Очистка данных

1. Удаляем пробелы в строковых значениях.
2. Преобразуем `year`, `selling_price`, `km_driven` в числовой тип.
3. Удаляем полные дубликаты и строки с пропусками.
4. Удаляем явно невалидные записи (неположительная цена, отрицательный пробег, недопустимый год).
5. Сохраняем прежние названия и порядок **всех восьми столбцов**.

Мы **не удаляем** дорогие автомобили только за высокую стоимость: они могут быть реальными наблюдениями.

In [ ]:
initial_rows = len(df)
clean = df.copy()

text_cols = ['name', 'fuel', 'seller_type', 'transmission', 'owner']
number_cols = ['year', 'selling_price', 'km_driven']

for feature in text_cols:
    clean[feature] = clean[feature].astype('string').str.strip().replace('', pd.NA)

for feature in number_cols:
    clean[feature] = pd.to_numeric(clean[feature], errors='coerce')

duplicate_count = int(clean.duplicated().sum())
clean = clean.drop_duplicates()

missing_row_count = int(clean.isna().any(axis=1).sum())
clean = clean.dropna()

valid = (
    clean['selling_price'].gt(0)
    & clean['km_driven'].ge(0)
    & clean['year'].between(1950, datetime.now().year)
    & clean['year'].mod(1).eq(0)
    & clean['km_driven'].mod(1).eq(0)
)
invalid_row_count = int((~valid).sum())
clean = clean.loc[valid].copy()

clean['year'] = clean['year'].astype('int64')
clean['km_driven'] = clean['km_driven'].astype('int64')
for feature in text_cols:
    clean[feature] = clean[feature].astype('category')
clean = clean.reset_index(drop=True)

assert tuple(clean.columns) == SOURCE_COLUMNS, 'Названия или порядок столбцов изменились!'
assert clean.shape[1] == 8
assert clean.notna().all().all(), 'Остались пропущенные значения!'
assert len(clean) >= 300, 'После очистки осталось менее 300 автомобилей!'

print('До очистки:', initial_rows)
print('Удалено полных дубликатов:', duplicate_count)
print('Удалено строк с пропусками:', missing_row_count)
print('Удалено строк с невалидными значениями:', invalid_row_count)
print('После очистки:', len(clean))
print('Столбцов:', clean.shape[1])
print('Типы данных:')
print(clean.dtypes)
display(clean.head())

## 3. Анализ признаков — 5 графиков

По методичке нужны **не менее пяти графиков, один интерактивный**, и **вывод после каждого графика**.

1. Распределение стоимости автомобилей.
2. Изменение медианной стоимости по году выпуска.
3. Стоимость по типу трансмиссии.
4. Стоимость по категории предыдущих владельцев.
5. Интерактивная зависимость цены от пробега с выделением типа топлива.

Цена в датасете указана в индийских рупиях (INR). Графики 1–4 сохраняются как PNG, график 5 — как HTML.

In [ ]:
# ГРАФИК 1 — распределение цен
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(
    data=clean, x='selling_price', bins=40,
    log_scale=True, color='steelblue', ax=ax
)
ax.set_title('Распределение стоимости автомобилей')
ax.set_xlabel('Цена продажи, INR (логарифмическая шкала)')
ax.set_ylabel('Количество автомобилей')
fig.tight_layout()
fig.savefig(EDA_DIR / 'graph1.png', bbox_inches='tight')
plt.show()

price_mean = float(clean['selling_price'].mean())
price_median = float(clean['selling_price'].median())
price_min = float(clean['selling_price'].min())
price_max = float(clean['selling_price'].max())

display(Markdown(
    f'**Вывод 1.** Средняя цена — **{price_mean:,.0f} INR**, медианная — '
    f'**{price_median:,.0f} INR**; диапазон цен — от **{price_min:,.0f}** '
    f'до **{price_max:,.0f} INR**. '
    + ('Средняя выше медианной: дорогие автомобили могут повышать среднее.'
       if price_mean > price_median else 'Средняя не превышает медианную.')
))

In [ ]:
# ГРАФИК 2 — медианная цена и год выпуска
price_by_year = clean.groupby('year')['selling_price'].median().sort_index()
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(price_by_year.index, price_by_year.values, marker='o', markersize=4)
ax.set_title('Медианная стоимость автомобилей по году выпуска')
ax.set_xlabel('Год выпуска')
ax.set_ylabel('Медианная стоимость, INR')
fig.tight_layout()
fig.savefig(EDA_DIR / 'graph2.png', bbox_inches='tight')
plt.show()

year_corr = float(clean['year'].corr(clean['selling_price'], method='pearson'))
year_direction = ('положительная' if year_corr > 0 else 'отрицательная' if year_corr < 0 else 'нулевая')
display(Markdown(
    f'**Вывод 2.** Коэффициент корреляции Пирсона между годом выпуска и ценой: '
    f'**{year_corr:.3f}** ({year_direction} линейная связь). '
    'На графике представлены медианы по годам, а не цены отдельных автомобилей. '
    'Корреляция не доказывает причинную связь.'
))

In [ ]:
# ГРАФИК 3 — тип трансмиссии и цена
fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=clean, x='transmission', y='selling_price',
    showfliers=False, ax=ax
)
ax.set_yscale('log')
ax.set_title('Стоимость автомобилей по типу трансмиссии')
ax.set_xlabel('Тип трансмиссии')
ax.set_ylabel('Цена, INR (логарифмическая шкала)')
fig.tight_layout()
fig.savefig(EDA_DIR / 'graph3.png', bbox_inches='tight')
plt.show()

trans_summary = clean.groupby('transmission', observed=True)['selling_price'].agg(['count', 'median'])
trans_summary = trans_summary.sort_values('median', ascending=False)
display(trans_summary.rename(columns={'count':'Количество', 'median':'Медианная цена, INR'}))
trans_medians = trans_summary['median']
display(Markdown(
    f'**Вывод 3.** Наибольшая медианная стоимость в категории '
    f'**{trans_medians.index[0]}** — **{trans_medians.iloc[0]:,.0f} INR**. '
    'Различия между группами могут быть связаны и с другими характеристиками автомобилей.'
))

In [ ]:
# ГРАФИК 4 — категория владельцев и цена
owner_summary = clean.groupby('owner', observed=True)['selling_price'].agg(['count', 'median'])
owner_summary = owner_summary.sort_values('median', ascending=False)
owner_medians = owner_summary['median']

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(owner_medians.index.astype(str), owner_medians.values, color='cornflowerblue')
ax.set_title('Медианная стоимость по категории владельцев')
ax.set_xlabel('Категория предыдущих владельцев')
ax.set_ylabel('Медианная стоимость, INR')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(EDA_DIR / 'graph4.png', bbox_inches='tight')
plt.show()

display(owner_summary.rename(columns={'count':'Количество', 'median':'Медианная цена, INR'}))
display(Markdown(
    f'**Вывод 4.** Самая высокая медианная стоимость — у категории '
    f'**{owner_medians.index[0]}** ({owner_medians.iloc[0]:,.0f} INR). '
    'Это описательное сравнение: при интерпретации необходимо учитывать число машин в каждой категории.'
))

In [ ]:
# ГРАФИК 5 — ИНТЕРАКТИВНАЯ цена от пробега и типа топлива
fig = px.scatter(
    clean,
    x='km_driven',
    y='selling_price',
    color='fuel',
    hover_data=['name', 'year', 'transmission', 'seller_type', 'owner'],
    log_y=True,
    opacity=0.6,
    title='Цена автомобиля в зависимости от пробега и типа топлива',
    labels={
        'km_driven': 'Пробег, км',
        'selling_price': 'Стоимость, INR',
        'fuel': 'Тип топлива'
    }
)
html_path = EDA_DIR / 'graph5_interactive.html'
fig.write_html(str(html_path), include_plotlyjs=True, full_html=True)
fig.show()

km_corr = float(clean['km_driven'].corr(clean['selling_price'], method='pearson'))
km_direction = ('положительная' if km_corr > 0 else 'отрицательная' if km_corr < 0 else 'нулевая')
fuel_counts = clean['fuel'].value_counts()
display(Markdown(
    f'**Вывод 5.** Коэффициент корреляции Пирсона между пробегом и ценой '
    f'равен **{km_corr:.3f}** ({km_direction} линейная связь). '
    f'Самая многочисленная категория топлива — **{fuel_counts.index[0]}** '
    f'({fuel_counts.iloc[0]} автомобилей). Интерактивность позволяет изучать отдельные '
    'наблюдения и сравнивать группы по топливу. Корреляция не означает причинность.'
))
print('Интерактивный график сохранён:', html_path)

## 4. Сохранение очищенных данных

Сохраняем те же 8 столбцов в `data/clean_dataset.pkl` для дальнейших лабораторных работ. **Файл `.pkl` нельзя добавлять в GitHub.**

In [ ]:
clean_path = PROJECT_ROOT / 'data' / 'clean_dataset.pkl'
clean.to_pickle(clean_path)

# Проверка корректности сохранения
restored = pd.read_pickle(clean_path)  # Открываем только созданный нами локальный файл.
pd.testing.assert_frame_equal(clean, restored)
assert tuple(restored.columns) == SOURCE_COLUMNS
print('Очищенный датасет сохранён:', clean_path)
print('Размер:', restored.shape)

## 5. Выводы и отчёт для README

В предыдущих ячейках выводы построены из рассчитанных данных. Ниже они собираются в файл `eda/eda_summary.md`. Перенесите этот текст в `README.md` и при необходимости дополните своими наблюдениями по графикам.

In [ ]:
# Формируем итоговый текст только после выполнения ячеек с графиками.
def median_lines(series):
    return '\n'.join(f'- {category}: {value:,.0f} INR' for category, value in series.items())

summary = f"""# Результаты разведочного анализа Car Price Prediction

## Исходные данные
- Тип задачи: регрессия.
- Целевая переменная: `selling_price` (INR).
- Количество объектов до очистки: {initial_rows}.
- Используемые столбцы ({len(SOURCE_COLUMNS)}): {', '.join(SOURCE_COLUMNS)}.

## Очистка данных
- Удалено полных дубликатов: {duplicate_count}.
- Удалено строк с пропусками: {missing_row_count}.
- Удалено строк с невалидными значениями: {invalid_row_count}.
- Объектов после очистки: {len(clean)}.
- Столбцов после очистки: {clean.shape[1]}.
- Новые признаки не создавались.

## Выводы по пяти графикам

### 1. Распределение стоимости
- Средняя цена: {price_mean:,.0f} INR.
- Медианная цена: {price_median:,.0f} INR.
- Минимум: {price_min:,.0f} INR; максимум: {price_max:,.0f} INR.

### 2. Год выпуска
- Корреляция Пирсона с ценой: {year_corr:.3f}.
- Сравнение медиан по годам выполнено на графике `graph2.png`.

### 3. Трансмиссия
{median_lines(trans_medians)}

### 4. Категория владельцев
{median_lines(owner_medians)}

### 5. Пробег и топливо (интерактивный график)
- Корреляция Пирсона между пробегом и ценой: {km_corr:.3f}.
- Самая частая категория топлива: {fuel_counts.index[0]} ({fuel_counts.iloc[0]} объектов).

## Результаты
- Статические графики: `eda/graph1.png`, `eda/graph2.png`, `eda/graph3.png`, `eda/graph4.png`.
- Интерактивный график: `eda/graph5_interactive.html` (хранить вне Git, если файл крупный).
- Очищенный датасет: `data/clean_dataset.pkl` (не добавлять в GitHub).

## Ограничения
Показанные взаимосвязи статистические и не доказывают причинно-следственной связи. При интерпретации категориальных групп важно учитывать численность наблюдений. Для обучения модели понадобится отдельная работа с кодированием категориальных признаков.
"""
summary_path = EDA_DIR / 'eda_summary.md'
summary_path.write_text(summary, encoding='utf-8')
display(Markdown(summary))
print('Отчёт создан:', summary_path)

## 6. Проверка перед сдачей

- [ ] Все ячейки выполняются без ошибок.
- [ ] В очищенной таблице **ровно 8 исходных столбцов**.
- [ ] Построены ровно **5 графиков**: `graph1.png`–`graph4.png` и `graph5_interactive.html`.
- [ ] После каждого графика сформирован вывод.
- [ ] `clean_dataset.pkl` сохранён отдельно от GitHub.
- [ ] В `README.md` добавлены результаты `eda_summary.md` и инструкция запуска.
- [ ] Перед коммитом очищены тяжёлые outputs блокнота (VS Code → **Clear All Outputs**).
- [ ] `.gitignore` включает `*.csv`, `*.pkl`, `.venv*/` и, при необходимости, `eda/*.html`.
- [ ] PNG-графики проверены на размер (не более 10 МБ каждый).

После проверки можно коммитить `eda.ipynb`, 4 PNG, `README.md` и `requirements.txt`.